# 07 Recommendation Confidence
Этап 13: top-k recommendations + confidence/uncertainty layer.

In [ ]:
import json
from pathlib import Path
import importlib
import numpy as np
import pandas as pd

from _shared_notebook_utils import RESEARCH_CHECKPOINT_DIR

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
registry_path = ROOT / "artifacts" / "model_registry.json"
with open(registry_path, "r", encoding="utf-8") as f:
    registry = json.load(f)

active_model = registry["active_model"]
active_entry = registry["models"][active_model]
model_path = ROOT / active_entry["model_path"]
meta_path = ROOT / active_entry["meta_path"]

if not model_path.exists() or not meta_path.exists():
    raise FileNotFoundError(f"Model artifacts missing for active_model={active_model}: {model_path} / {meta_path}")

# Load split dataframes from frozen checkpoint.
baseline_dir = RESEARCH_CHECKPOINT_DIR / "baseline"
val_df = pd.read_pickle(baseline_dir / "val_df.pkl")
test_df = pd.read_pickle(baseline_dir / "test_df.pkl")

with open(meta_path, "r", encoding="utf-8") as f:
    model_meta = json.load(f)

catboost_mod = importlib.import_module("catboost")
CatBoostClassifier = catboost_mod.CatBoostClassifier
model = CatBoostClassifier()
model.load_model(model_path.as_posix())

print("active_model =", active_model)
print("Loaded model:", model_path)
print("Loaded split shapes:", val_df.shape, test_df.shape)

In [ ]:
# 7.1 Recommendation layer + confidence/uncertainty
feature_columns = model_meta.get("feature_columns", model_meta.get("feature_cols", []))
if not feature_columns:
    raise ValueError("feature columns are missing in model meta")

if "id_to_target" in model_meta:
    id_to_target = {int(k): str(v) for k, v in model_meta["id_to_target"].items()}
    class_names = np.asarray([id_to_target[i] for i in sorted(id_to_target.keys())], dtype=object)
elif "target_to_id" in model_meta:
    inv = {int(v): str(k) for k, v in model_meta["target_to_id"].items()}
    class_names = np.asarray([inv[i] for i in sorted(inv.keys())], dtype=object)
else:
    raise ValueError("No class mapping found in model meta")

for split_df in [val_df, test_df]:
    for col in ["history_1", "history_2", "history_3", "target"]:
        if col in split_df.columns:
            split_df[col] = split_df[col].astype("string")

X_val = val_df[feature_columns]
X_test = test_df[feature_columns]
val_proba = np.asarray(model.predict_proba(X_val))
test_proba = np.asarray(model.predict_proba(X_test))

if val_proba.shape[1] != len(class_names):
    raise ValueError(f"Class/proba mismatch for validation: {val_proba.shape[1]} vs {len(class_names)}")
if test_proba.shape[1] != len(class_names):
    raise ValueError(f"Class/proba mismatch for test: {test_proba.shape[1]} vs {len(class_names)}")

def extract_topk_arrays(proba_matrix, class_names_arr, max_k=5):
    max_k = min(int(max_k), proba_matrix.shape[1])
    top_idx_unsorted = np.argpartition(proba_matrix, -max_k, axis=1)[:, -max_k:]
    top_proba_unsorted = np.take_along_axis(proba_matrix, top_idx_unsorted, axis=1)
    sort_order = np.argsort(-top_proba_unsorted, axis=1)
    top_idx = np.take_along_axis(top_idx_unsorted, sort_order, axis=1)
    top_proba = np.take_along_axis(top_proba_unsorted, sort_order, axis=1)
    top_labels = class_names_arr[top_idx]
    return top_idx, top_proba, top_labels

def build_recommendation_df(split_df, top_labels, top_proba):
    rec_df = split_df[[c for c in ["history_1", "history_2", "history_3", "target"] if c in split_df.columns]].copy()
    rec_df["top1_pred"] = top_labels[:, 0]
    rec_df["top1_proba"] = top_proba[:, 0].astype(float)
    rec_df["top2_pred"] = top_labels[:, 1]
    rec_df["top2_proba"] = top_proba[:, 1].astype(float)
    rec_df["top3_pred"] = top_labels[:, 2]
    rec_df["top3_proba"] = top_proba[:, 2].astype(float)
    rec_df["top3_list"] = top_labels[:, 0].astype(str) + " | " + top_labels[:, 1].astype(str) + " | " + top_labels[:, 2].astype(str)
    rec_df["top5_list"] = (
        top_labels[:, 0].astype(str) + " | " + top_labels[:, 1].astype(str) + " | " + top_labels[:, 2].astype(str)
        + " | " + top_labels[:, 3].astype(str) + " | " + top_labels[:, 4].astype(str)
    )

    y_true = split_df["target"].astype("string").to_numpy(dtype=object)
    rec_df["is_top1_correct"] = (top_labels[:, 0] == y_true)
    rec_df["is_hit_at_3"] = np.any(top_labels[:, :3] == y_true[:, None], axis=1)
    rec_df["is_hit_at_5"] = np.any(top_labels[:, :5] == y_true[:, None], axis=1)
    return rec_df

val_top_idx, val_top_proba, val_top_labels = extract_topk_arrays(val_proba, class_names, max_k=5)
test_top_idx, test_top_proba, test_top_labels = extract_topk_arrays(test_proba, class_names, max_k=5)

rec_val_df = build_recommendation_df(val_df, val_top_labels, val_top_proba)
rec_test_df = build_recommendation_df(test_df, test_top_labels, test_top_proba)

def compute_topk_summary(rec_df, split_name):
    return {
        "model": f"catboost_recommendation_{active_model}",
        "split": split_name,
        "accuracy_at_1": float(rec_df["is_top1_correct"].mean()),
        "hit_at_3": float(rec_df["is_hit_at_3"].mean()),
        "hit_at_5": float(rec_df["is_hit_at_5"].mean()),
    }

metrics_recommendation_df = pd.DataFrame([
    compute_topk_summary(rec_val_df, "validation"),
    compute_topk_summary(rec_test_df, "test"),
])

n_classes = len(class_names)
for rec_df, proba in [(rec_val_df, val_proba), (rec_test_df, test_proba)]:
    rec_df["margin_top1_top2"] = rec_df["top1_proba"] - rec_df["top2_proba"]
    rec_df["entropy"] = (-np.sum(proba * np.log(proba + 1e-12), axis=1) / np.log(n_classes)).astype(float)
    high_mask = (rec_df["top1_proba"] >= 0.70) & (rec_df["margin_top1_top2"] >= 0.25)
    medium_mask = (rec_df["top1_proba"] >= 0.50) & (rec_df["margin_top1_top2"] >= 0.12) & (~high_mask)
    rec_df["uncertainty_bucket"] = np.select(
        [high_mask, medium_mask],
        ["high_confidence", "medium_confidence"],
        default="low_confidence",
    )

def confidence_summary_table(rec_df, split_name):
    out = (
        rec_df.groupby("uncertainty_bucket", dropna=False)
        .agg(
            n_samples=("uncertainty_bucket", "size"),
            accuracy_at_1=("is_top1_correct", "mean"),
            hit_at_3=("is_hit_at_3", "mean"),
        )
        .reset_index()
    )
    out.insert(0, "split", split_name)
    return out

confidence_summary_df = pd.concat([
    confidence_summary_table(rec_val_df, "validation"),
    confidence_summary_table(rec_test_df, "test"),
], ignore_index=True)

def per_class_topk_table(rec_df, split_name):
    out = (
        rec_df.groupby("target", dropna=False)
        .agg(
            support=("target", "size"),
            recall_at_1=("is_top1_correct", "mean"),
            recall_at_3=("is_hit_at_3", "mean"),
            recall_at_5=("is_hit_at_5", "mean"),
        )
        .reset_index()
        .rename(columns={"target": "class_name"})
    )
    out.insert(0, "split", split_name)
    return out

per_class_topk_df = pd.concat([
    per_class_topk_table(rec_val_df, "validation"),
    per_class_topk_table(rec_test_df, "test"),
], ignore_index=True)

print("Table 1. Overall metrics")
display(metrics_recommendation_df)
print("Table 2. Confidence summary")
display(confidence_summary_df)
print("Table 3. Per-class top-k metrics")
display(per_class_topk_df.sort_values(["split", "support"], ascending=[True, False]))

sample_cols = [
    c for c in [
        "history_1", "history_2", "history_3", "target",
        "top1_pred", "top1_proba", "top2_pred", "top2_proba", "top3_pred", "top3_proba",
        "top3_list", "top5_list", "margin_top1_top2", "entropy", "uncertainty_bucket", "is_hit_at_3",
    ] if c in rec_val_df.columns
]

print("Validation sample:")
display(rec_val_df[sample_cols].sample(n=min(15, len(rec_val_df)), random_state=42))
print("Test sample:")
display(rec_test_df[sample_cols].sample(n=min(15, len(rec_test_df)), random_state=42))

results_dir = ROOT / "artifacts" / "results" / "recommendation"
results_dir.mkdir(parents=True, exist_ok=True)
metrics_recommendation_df.to_csv(results_dir / f"recommendation_metrics_{active_model}.csv", index=False)
confidence_summary_df.to_csv(results_dir / f"confidence_summary_{active_model}.csv", index=False)
per_class_topk_df.to_csv(results_dir / f"per_class_topk_{active_model}.csv", index=False)
rec_val_df[sample_cols].head(5000).to_csv(results_dir / f"rec_val_preview_{active_model}.csv", index=False)
rec_test_df[sample_cols].head(5000).to_csv(results_dir / f"rec_test_preview_{active_model}.csv", index=False)
print("Saved recommendation artifacts to:", results_dir)